<a href="https://colab.research.google.com/github/smonodeep/DynamicValuationGovernance/blob/main/0.DBA_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# Load HPI data
df_hpi = pd.read_csv('/content/rbi_hpi_chainlinked.csv')

# Clean column names
df_hpi.columns = [col.lower().strip() for col in df_hpi.columns]

# Normalize city names and remove symbols
# The raw 'city' column already contains full names, so we just remove asterisks.
df_hpi['city'] = df_hpi['city'].str.replace('*', '', regex=False)

# Standardize quarter format - The current raw format 'Q# YYYY' is already handled by the function.
def standardize_rbi_quarter(q_str):
    if 'Q' in q_str and '.' in q_str and '-' in q_str:
        parts = q_str.split('.')
        q_num = parts[0]
        year_part = parts[1].split('-')[1]
        return f"{q_num} 20{year_part}"
    return q_str

df_hpi['quarter'] = df_hpi['quarter'].apply(standardize_rbi_quarter)

# Rename 'hpi_chainlinked' to 'hpi' for consistency with comparison_df
df_hpi.rename(columns={'hpi_chainlinked': 'hpi'}, inplace=True)

df_hpi_cleaned = df_hpi.copy()
print("HPI Data (first 5 rows after cleaning and standardization):")
display(df_hpi_cleaned.head())

HPI Data (first 5 rows after cleaning and standardization):


,quarter,city,hpi
0,Q1 2010,ahmedabad,64.754759
1,Q1 2011,ahmedabad,31.937650
2,Q1 2012,ahmedabad,41.557409
3,Q1 2013,ahmedabad,48.224069
4,Q1 2014,ahmedabad,55.482063


In [11]:
# Load Rent data
df_rent = pd.read_csv('/content/Rental_Index_CREMatrix.csv')

# Clean column names
df_rent.columns = [col.lower().strip() for col in df_rent.columns]

# Normalize city names
city_mapping = {"blr":"bangalore","chn":"chennai","del":"delhi","mum":"mumbai"}
df_rent['city'] = df_rent['city'].map(city_mapping)

# Standardize quarter format (assuming 'Q# YYYY' already, but robustify)
def standardize_quarter_format(q_str):
    if ' ' not in q_str and len(q_str) == 6 and q_str.startswith('Q'): # e.g., Q12020
        return f"{q_str[:2]} {q_str[2:]}"
    return q_str

df_rent['quarter'] = df_rent['quarter'].apply(standardize_quarter_format)

# Aggregate rent to (city, quarter) using mean
df_rent_agg = df_rent.groupby(['quarter', 'city'])['rent'].mean().reset_index()

df_rent_cleaned = df_rent_agg.copy()
print("Rent Data (first 5 rows after cleaning, aggregation, and standardization):")
display(df_rent_cleaned.head())

Rent Data (first 5 rows after cleaning, aggregation, and standardization):


,quarter,city,rent
0,Q1 2014,bangalore,100.00
1,Q1 2014,chennai,100.00
2,Q1 2014,delhi,100.00
3,Q1 2014,mumbai,100.00
4,Q1 2015,bangalore,104.45


In [12]:
# Ensure 'city' and 'quarter' columns are clean before merging
df_rent_cleaned['city'] = df_rent_cleaned['city'].str.strip()
df_rent_cleaned['quarter'] = df_rent_cleaned['quarter'].str.strip()
df_hpi_cleaned['city'] = df_hpi_cleaned['city'].str.strip()
df_hpi_cleaned['quarter'] = df_hpi_cleaned['quarter'].str.strip()

# Merge rent and HPI data
df_merged = pd.merge(df_rent_cleaned, df_hpi_cleaned, on=['quarter', 'city'], how='left')

# Sort by city, then quarter
df_merged['quarter_sort_key'] = df_merged['quarter'].str.replace('Q', '').str.replace(' ', '')
df_merged['quarter_sort_key'] = df_merged['quarter_sort_key'].astype(int)
df_merged = df_merged.sort_values(by=['city', 'quarter_sort_key']).drop(columns=['quarter_sort_key'])

print("Merged Data (first 5 rows):")
display(df_merged.head())

Merged Data (first 5 rows):


,quarter,city,rent,hpi
0,Q1 2014,bangalore,100.0000,48.741877
4,Q1 2015,bangalore,104.4500,61.806841
8,Q1 2016,bangalore,114.6000,71.406332
12,Q1 2017,bangalore,121.9875,75.582470
16,Q1 2018,bangalore,145.2250,77.997789


In [13]:
# Inspect the first few rows of the raw rbi_hpi_chainlinked.csv to understand its structure.
# This is done to diagnose why the 'city' column is all NaN after initial processing.
# The file is '/content/rbi_hpi_chainlinked.csv'.

print("Raw 'rbi_hpi_chainlinked.csv' head:")
raw_hpi_df = pd.read_csv('/content/rbi_hpi_chainlinked.csv')
display(raw_hpi_df.head(10))

Raw 'rbi_hpi_chainlinked.csv' head:


,quarter,city,hpi_chainlinked
0,Q1 2010,ahmedabad,64.754759
1,Q1 2011,ahmedabad,31.937650
2,Q1 2012,ahmedabad,41.557409
3,Q1 2013,ahmedabad,48.224069
4,Q1 2014,ahmedabad,55.482063
5,Q1 2015,ahmedabad,59.271527
6,Q1 2016,ahmedabad,63.914143
7,Q1 2017,ahmedabad,71.130236
8,Q1 2018,ahmedabad,83.288404
9,Q1 2019,ahmedabad,87.016614


### 2. Load Rent Data & Cleaning

- Load `Rental_Index_CREMatrix.csv`.
- Convert column names to lowercase and strip spaces.
- Normalize city names.
- Aggregate rent to `(city, quarter)` using mean.
- Standardize quarter format to `Q# YYYY`.

In [14]:
# Load Rent data
df_rent = pd.read_csv('/content/Rental_Index_CREMatrix.csv')

# Clean column names
df_rent.columns = [col.lower().strip() for col in df_rent.columns]

# Normalize city names
city_mapping = {"blr":"bangalore","chn":"chennai","del":"delhi","mum":"mumbai"}
df_rent['city'] = df_rent['city'].map(city_mapping)

# Standardize quarter format (assuming 'Q# YYYY' already, but robustify)
def standardize_quarter_format(q_str):
    if ' ' not in q_str and len(q_str) == 6 and q_str.startswith('Q'): # e.g., Q12020
        return f"{q_str[:2]} {q_str[2:]}"
    return q_str

df_rent['quarter'] = df_rent['quarter'].apply(standardize_quarter_format)

# Aggregate rent to (city, quarter) using mean
df_rent_agg = df_rent.groupby(['quarter', 'city'])['rent'].mean().reset_index()

df_rent_cleaned = df_rent_agg.copy()
print("Rent Data (first 5 rows after cleaning, aggregation, and standardization):")
display(df_rent_cleaned.head())

Rent Data (first 5 rows after cleaning, aggregation, and standardization):


,quarter,city,rent
0,Q1 2014,bangalore,100.00
1,Q1 2014,chennai,100.00
2,Q1 2014,delhi,100.00
3,Q1 2014,mumbai,100.00
4,Q1 2015,bangalore,104.45


### 3. Merge Rent and HPI Data

- Perform a `LEFT JOIN` (rent -> HPI) on `city` and `quarter`.
- Sort the merged data by `city`, then `quarter`.

In [15]:
# Merge rent and HPI data
df_merged = pd.merge(df_rent_cleaned, df_hpi_cleaned, on=['quarter', 'city'], how='left')

# Sort by city, then quarter
df_merged['quarter_sort_key'] = df_merged['quarter'].str.replace('Q', '').str.replace(' ', '')
df_merged['quarter_sort_key'] = df_merged['quarter_sort_key'].astype(int)
df_merged = df_merged.sort_values(by=['city', 'quarter_sort_key']).drop(columns=['quarter_sort_key'])

print("Merged Data (first 5 rows):")
display(df_merged.head())

Merged Data (first 5 rows):


,quarter,city,rent,hpi
0,Q1 2014,bangalore,100.0000,48.741877
4,Q1 2015,bangalore,104.4500,61.806841
8,Q1 2016,bangalore,114.6000,71.406332
12,Q1 2017,bangalore,121.9875,75.582470
16,Q1 2018,bangalore,145.2250,77.997789


### 4. Growth Rate Calculation & Valuation Gap

- Compute `rent_growth` and `hpi_growth` using log differences, grouped by `city`.
- Calculate `valuation_gap = rent_growth - hpi_growth`.

In [16]:
# Calculate growth rates (log difference)
df_merged['rent_growth'] = df_merged.groupby('city')['rent'].transform(lambda x: np.log(x).diff())
df_merged['hpi_growth'] = df_merged.groupby('city')['hpi'].transform(lambda x: np.log(x).diff())

# Calculate valuation gap
df_merged['valuation_gap'] = df_merged['rent_growth'] - df_merged['hpi_growth']

print("Data with Growth Rates and Valuation Gap (first 5 rows):")
display(df_merged.head())

Data with Growth Rates and Valuation Gap (first 5 rows):


,quarter,city,rent,hpi,rent_growth,hpi_growth,valuation_gap
0,Q1 2014,bangalore,100.0000,48.741877,NaN,NaN,NaN
4,Q1 2015,bangalore,104.4500,61.806841,0.043538,0.237475,-0.193937
8,Q1 2016,bangalore,114.6000,71.406332,0.092739,0.144372,-0.051633
12,Q1 2017,bangalore,121.9875,75.582470,0.062471,0.056838,0.005633
16,Q1 2018,bangalore,145.2250,77.997789,0.174366,0.031456,0.142910


### 5. Fetch and Process FRED Repo Data

- Fetch `INDIRLTLT01STM` series from FRED using `pandas_datareader`.
- Convert daily data to quarterly average.
- Create a `quarter` column in `Q# YYYY` format.

In [17]:
# Fetch FRED repo rate data
# start_date = df_merged['quarter'].min().split(' ')[1]
# end_date = df_merged['quarter'].max().split(' ')[1]

# User specified fixed dates
start_date_fixed = datetime.datetime(2010,1,1)
end_date_fixed = datetime.datetime(2025,1,1)

# Use pandas_datareader to fetch the series
repo_rate_series = pdr.get_data_fred('INDIRLTLT01STM',
                                       start=start_date_fixed,
                                       end=end_date_fixed)

# Convert to DataFrame and reset index
df_repo = repo_rate_series.reset_index()
df_repo.columns = ['date', 'repo_rate']

# Convert daily data to quarterly average
df_repo['quarter'] = df_repo['date'].dt.to_period('Q').astype(str)

# Map period format (YYYYQ[1-4]) to Q# YYYY
def format_quarter_from_period(q_period_str):
    year = q_period_str[:4]
    q_num = q_period_str[5:]
    return f"Q{q_num} {year}"

df_repo['quarter'] = df_repo['quarter'].apply(format_quarter_from_period)

df_repo_agg = df_repo.groupby('quarter')['repo_rate'].mean().reset_index()

print("Repo Rate Data (first 5 rows after processing):")
display(df_repo_agg.head())

Repo Rate Data (first 5 rows after processing):


,quarter,repo_rate
0,Q1 2012,8.267167
1,Q1 2013,7.873333
2,Q1 2014,8.804847
3,Q1 2015,7.782887
4,Q1 2016,7.657853


### 6. Merge Repo Data

- Perform a `LEFT JOIN` on `quarter` to merge the repo rate into the main dataset.

In [18]:
# Merge repo rate data
final_df = pd.merge(df_merged, df_repo_agg, on='quarter', how='left')

print("Data with Repo Rate (first 5 rows):")
display(final_df.head())

Data with Repo Rate (first 5 rows):


,quarter,city,rent,hpi,rent_growth,hpi_growth,valuation_gap,repo_rate
0,Q1 2014,bangalore,100.0000,48.741877,NaN,NaN,NaN,8.804847
1,Q1 2015,bangalore,104.4500,61.806841,0.043538,0.237475,-0.193937,7.782887
2,Q1 2016,bangalore,114.6000,71.406332,0.092739,0.144372,-0.051633,7.657853
3,Q1 2017,bangalore,121.9875,75.582470,0.062471,0.056838,0.005633,6.962920
4,Q1 2018,bangalore,145.2250,77.997789,0.174366,0.031456,0.142910,7.466353


### 7. Final Dataset Cleaning & Validation

- Drop rows where `rent_growth` OR `hpi_growth` is NaN.
- Print dataset shape, missing values per column, and unique cities.
- Ensure the dataset is not empty.

In [19]:
# Drop rows where rent_growth OR hpi_growth is NaN
# This ensures we only have complete growth observations
final_model_dataset = final_df.dropna(subset=['rent_growth', 'hpi_growth'])

# Select and reorder final columns as specified
final_model_dataset = final_model_dataset[[
    'city', 'quarter', 'rent', 'hpi',
    'rent_growth', 'hpi_growth', 'valuation_gap', 'repo_rate'
]]

# --- Validation ---
print("\n--- Dataset Validation ---")

# Dataset shape
print(f"Dataset Shape: {final_model_dataset.shape}")

# Missing values per column
print("\nMissing values per column:")
print(final_model_dataset.isnull().sum())

# Unique cities
print("\nUnique Cities:")
print(final_model_dataset['city'].unique())

# Ensure dataset is not empty
if final_model_dataset.empty:
    print("\n Warning: The final dataset is empty!")
else:
    print("\n Dataset is not empty.")


--- Dataset Validation ---
Dataset Shape: (184, 8)

Missing values per column:
city             0
quarter          0
rent             0
hpi              0
rent_growth      0
hpi_growth       0
valuation_gap    0
repo_rate        8
dtype: int64

Unique Cities:
['bangalore' 'chennai' 'delhi' 'mumbai']

 Dataset is not empty.


### 8. Bonus: Add Lag Variables

- Add `hpi_lag = lag(hpi_growth, 1)`
- Add `rent_lag = lag(rent_growth, 1)`

In [20]:
# Add lag variables
final_model_dataset['hpi_lag'] = final_model_dataset.groupby('city')['hpi_growth'].shift(1)
final_model_dataset['rent_lag'] = final_model_dataset.groupby('city')['rent_growth'].shift(1)

print("Final Dataset with Lag Variables (first 5 rows):")
display(final_model_dataset.head())

Final Dataset with Lag Variables (first 5 rows):


,city,quarter,rent,hpi,rent_growth,hpi_growth,valuation_gap,repo_rate,hpi_lag,rent_lag
1,bangalore,Q1 2015,104.4500,61.806841,0.043538,0.237475,-0.193937,7.782887,NaN,NaN
2,bangalore,Q1 2016,114.6000,71.406332,0.092739,0.144372,-0.051633,7.657853,0.237475,0.043538
3,bangalore,Q1 2017,121.9875,75.582470,0.062471,0.056838,0.005633,6.962920,0.144372,0.092739
4,bangalore,Q1 2018,145.2250,77.997789,0.174366,0.031456,0.142910,7.466353,0.056838,0.062471
5,bangalore,Q1 2019,144.7375,83.885498,-0.003363,0.072772,-0.076135,7.397403,0.031456,0.174366


In [21]:
final_model_dataset

,city,quarter,rent,hpi,rent_growth,hpi_growth,valuation_gap,repo_rate,hpi_lag,rent_lag
1,bangalore,Q1 2015,104.450000,61.806841,0.043538,0.237475,-0.193937,7.782887,NaN,NaN
2,bangalore,Q1 2016,114.600000,71.406332,0.092739,0.144372,-0.051633,7.657853,0.237475,0.043538
3,bangalore,Q1 2017,121.987500,75.582470,0.062471,0.056838,0.005633,6.962920,0.144372,0.092739
4,bangalore,Q1 2018,145.225000,77.997789,0.174366,0.031456,0.142910,7.466353,0.056838,0.062471
5,bangalore,Q1 2019,144.737500,83.885498,-0.003363,0.072772,-0.076135,7.397403,0.031456,0.174366
...,...,...,...,...,...,...,...,...,...,...
183,mumbai,Q4 2020,110.657143,90.592899,-0.010147,0.011342,-0.021489,5.910980,0.016050,0.063304
184,mumbai,Q4 2021,108.457143,91.307325,-0.020082,0.007855,-0.027937,6.384825,0.011342,-0.010147
185,mumbai,Q4 2022,114.485714,96.374888,0.054095,0.054015,0.000080,7.372220,0.007855,-0.020082
186,mumbai,Q4 2023,124.600000,100.695009,0.084659,0.043851,0.040808,7.281333,0.054015,0.054095


In [22]:
print("Final DataFrame *before* dropping NaNs in growth rates:")
display(final_df)

Final DataFrame *before* dropping NaNs in growth rates:


,quarter,city,rent,hpi,rent_growth,hpi_growth,valuation_gap,repo_rate
0,Q1 2014,bangalore,100.000000,48.741877,NaN,NaN,NaN,8.804847
1,Q1 2015,bangalore,104.450000,61.806841,0.043538,0.237475,-0.193937,7.782887
2,Q1 2016,bangalore,114.600000,71.406332,0.092739,0.144372,-0.051633,7.657853
3,Q1 2017,bangalore,121.987500,75.582470,0.062471,0.056838,0.005633,6.962920
4,Q1 2018,bangalore,145.225000,77.997789,0.174366,0.031456,0.142910,7.466353
...,...,...,...,...,...,...,...,...
183,Q4 2020,mumbai,110.657143,90.592899,-0.010147,0.011342,-0.021489,5.910980
184,Q4 2021,mumbai,108.457143,91.307325,-0.020082,0.007855,-0.027937,6.384825
185,Q4 2022,mumbai,114.485714,96.374888,0.054095,0.054015,0.000080,7.372220
186,Q4 2023,mumbai,124.600000,100.695009,0.084659,0.043851,0.040808,7.281333


As you can see above, `final_df` initially contained data for Q1 2020 and Q1 2021, but these rows had `NaN` values in the `rent_growth` and `hpi_growth` columns. The `dropna` step removed them. If you wish to retain these rows, you would need to adjust the `dropna` condition or decide how to fill those `NaN` values (e.g., with 0, or by using a different imputation method).

### Detailed Comparison of `final_model_dataset` and `comparison_df`

In [24]:
print("Columns in final_model_dataset:")
display(final_model_dataset.columns.tolist())

print("\nColumns in comparison_df:")
display(comparison_df.columns.tolist())

Columns in final_model_dataset:


['city',
 'quarter',
 'rent',
 'hpi',
 'rent_growth',
 'hpi_growth',
 'valuation_gap',
 'repo_rate',
 'hpi_lag',
 'rent_lag']


Columns in comparison_df:


['city',
 'quarter',
 'rent',
 'hpi',
 'hpi_growth',
 'hpi_norm',
 'repo_rate',
 'date']

In [25]:
common_cols = list(set(final_model_dataset.columns) & set(comparison_df.columns))
unique_to_final = list(set(final_model_dataset.columns) - set(comparison_df.columns))
unique_to_comparison = list(set(comparison_df.columns) - set(final_model_dataset.columns))

print(f"Common columns: {common_cols}")
print(f"Columns unique to final_model_dataset: {unique_to_final}")
print(f"Columns unique to comparison_df: {unique_to_comparison}")

Common columns: ['rent', 'hpi_growth', 'hpi', 'city', 'quarter', 'repo_rate']
Columns unique to final_model_dataset: ['valuation_gap', 'rent_lag', 'hpi_lag', 'rent_growth']
Columns unique to comparison_df: ['date', 'hpi_norm']


In [26]:
print("Data types for common columns in final_model_dataset:")
display(final_model_dataset[common_cols].dtypes)

print("\nData types for common columns in comparison_df:")
display(comparison_df[common_cols].dtypes)

Data types for common columns in final_model_dataset:


,0
rent,float64
hpi_growth,float64
hpi,float64
city,object
quarter,object
repo_rate,float64



Data types for common columns in comparison_df:


,0
rent,float64
hpi_growth,float64
hpi,float64
city,object
quarter,object
repo_rate,float64


The previous `dropna` and `shift` operations on `final_model_dataset` would have impacted the time period covered. Let's look at the unique quarters in both dataframes.

In [27]:
print("Unique quarters in final_model_dataset:")
display(final_model_dataset['quarter'].sort_values().unique())

print("\nUnique quarters in comparison_df (first 10):")
display(comparison_df['quarter'].sort_values().unique()[:10])

print("\nMin/Max quarter in final_model_dataset:")
print(f"Min: {final_model_dataset['quarter'].min()}, Max: {final_model_dataset['quarter'].max()}")

print("\nMin/Max quarter in comparison_df:")
print(f"Min: {comparison_df['quarter'].min()}, Max: {comparison_df['quarter'].max()}")

Unique quarters in final_model_dataset:


array(['Q1 2015', 'Q1 2016', 'Q1 2017', 'Q1 2018', 'Q1 2019', 'Q1 2020',
       'Q1 2021', 'Q1 2022', 'Q1 2023', 'Q1 2024', 'Q1 2025', 'Q2 2014',
       'Q2 2015', 'Q2 2016', 'Q2 2017', 'Q2 2018', 'Q2 2019', 'Q2 2020',
       'Q2 2021', 'Q2 2022', 'Q2 2023', 'Q2 2024', 'Q2 2025', 'Q3 2014',
       'Q3 2015', 'Q3 2016', 'Q3 2017', 'Q3 2018', 'Q3 2019', 'Q3 2020',
       'Q3 2021', 'Q3 2022', 'Q3 2023', 'Q3 2024', 'Q3 2025', 'Q4 2014',
       'Q4 2015', 'Q4 2016', 'Q4 2017', 'Q4 2018', 'Q4 2019', 'Q4 2020',
       'Q4 2021', 'Q4 2022', 'Q4 2023', 'Q4 2024'], dtype=object)


Unique quarters in comparison_df (first 10):


array(['Q1 2014', 'Q1 2015', 'Q1 2016', 'Q1 2017', 'Q1 2018', 'Q1 2019',
       'Q1 2020', 'Q1 2021', 'Q1 2022', 'Q1 2023'], dtype=object)


Min/Max quarter in final_model_dataset:
Min: Q1 2015, Max: Q4 2024

Min/Max quarter in comparison_df:
Min: Q1 2014, Max: Q4 2024


In [28]:
print("Missing values in final_model_dataset:")
display(final_model_dataset.isnull().sum())

Missing values in final_model_dataset:


,0
city,0
quarter,0
rent,0
hpi,0
rent_growth,0
hpi_growth,0
valuation_gap,0
repo_rate,8
hpi_lag,4
rent_lag,4


In [30]:
import pandas as pd

# Load the comparison CSV file
try:
    comparison_df = pd.read_csv('/content/merged_hpi_rent_repo.csv')
    print("Successfully loaded 'merged_hpi_rent_repo.csv'.")
    print("Shape of comparison_df:", comparison_df.shape)
    print("First 5 rows of comparison_df:")
    display(comparison_df.head())

    print("\n--- Comparing DataFrames ---")
    if final_model_dataset.equals(comparison_df):
        print("The final_model_dataset and merged_hpi_rent_repo.csv are identical.")
    else:
        print("The two DataFrames are different. Let's examine the differences:")

        # Check for differences in shape
        if final_model_dataset.shape != comparison_df.shape:
            print(f"- Shapes differ: final_model_dataset {final_model_dataset.shape} vs comparison_df {comparison_df.shape}")

        # Find rows that are in comparison_df but not in final_model_dataset
        merged_outer = pd.merge(final_model_dataset, comparison_df, how='outer', indicator=True)
        diff_in_comparison = merged_outer[merged_outer['_merge'] == 'right_only']

        if not diff_in_comparison.empty:
            print("\n- Rows present in 'merged_hpi_rent_repo.csv' but not in 'final_model_dataset':")
            display(diff_in_comparison.drop(columns='_merge').head())
        else:
            print("\n- No rows found in 'merged_hpi_rent_repo.csv' that are missing in 'final_model_dataset'.")

        # Find rows that are in final_model_dataset but not in comparison_df
        diff_in_final = merged_outer[merged_outer['_merge'] == 'left_only']

        if not diff_in_final.empty:
            print("\n- Rows present in 'final_model_dataset' but not in 'merged_hpi_rent_repo.csv':")
            display(diff_in_final.drop(columns='_merge').head())
        else:
            print("\n- No rows found in 'final_model_dataset' that are missing in 'merged_hpi_rent_repo.csv'.")

except FileNotFoundError:
    print("Error: 'merged_hpi_rent_repo.csv' not found. Please upload the file to the /content/ directory.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded 'merged_hpi_rent_repo.csv'.
Shape of comparison_df: (232, 8)
First 5 rows of comparison_df:


,city,quarter,rent,hpi,hpi_growth,hpi_norm,repo_rate,date
0,bangalore,Q1 2014,100.0000,142.266366,0.064893,1.373227,8.804847,2014-01-01
1,bangalore,Q2 2014,108.2750,150.373377,0.096215,1.451480,8.791340,2014-04-01
2,bangalore,Q3 2014,99.5375,169.256843,0.181414,1.633753,8.569230,2014-07-01
3,bangalore,Q4 2014,109.8875,184.285621,0.217250,1.778819,8.181552,2014-10-01
4,bangalore,Q1 2015,104.4500,180.400000,0.237475,1.741313,7.782887,2015-01-01



--- Comparing DataFrames ---
The two DataFrames are different. Let's examine the differences:
- Shapes differ: final_model_dataset (184, 10) vs comparison_df (232, 8)

- Rows present in 'merged_hpi_rent_repo.csv' but not in 'final_model_dataset':


,city,quarter,rent,hpi,rent_growth,hpi_growth,valuation_gap,repo_rate,hpi_lag,rent_lag,hpi_norm,date
0,bangalore,Q1 2014,100.0000,142.266366,NaN,0.064893,NaN,8.804847,NaN,NaN,1.373227,2014-01-01
2,bangalore,Q1 2015,104.4500,180.400000,NaN,0.237475,NaN,7.782887,NaN,NaN,1.741313,2015-01-01
4,bangalore,Q1 2016,114.6000,208.418714,NaN,0.144372,NaN,7.657853,NaN,NaN,2.011764,2016-01-01
6,bangalore,Q1 2017,121.9875,220.607905,NaN,0.056838,NaN,6.962920,NaN,NaN,2.129420,2017-01-01
8,bangalore,Q1 2018,145.2250,227.657667,NaN,0.031456,NaN,7.466353,NaN,NaN,2.197468,2018-01-01



- Rows present in 'final_model_dataset' but not in 'merged_hpi_rent_repo.csv':


,city,quarter,rent,hpi,rent_growth,hpi_growth,valuation_gap,repo_rate,hpi_lag,rent_lag,hpi_norm,date
1,bangalore,Q1 2015,104.4500,61.806841,0.043538,0.237475,-0.193937,7.782887,NaN,NaN,NaN,NaN
3,bangalore,Q1 2016,114.6000,71.406332,0.092739,0.144372,-0.051633,7.657853,0.237475,0.043538,NaN,NaN
5,bangalore,Q1 2017,121.9875,75.582470,0.062471,0.056838,0.005633,6.962920,0.144372,0.092739,NaN,NaN
7,bangalore,Q1 2018,145.2250,77.997789,0.174366,0.031456,0.142910,7.466353,0.056838,0.062471,NaN,NaN
9,bangalore,Q1 2019,144.7375,83.885498,-0.003363,0.072772,-0.076135,7.397403,0.031456,0.174366,NaN,NaN


In [31]:
display(final_model_dataset.head())

,city,quarter,rent,hpi,rent_growth,hpi_growth,valuation_gap,repo_rate,hpi_lag,rent_lag
1,bangalore,Q1 2015,104.4500,61.806841,0.043538,0.237475,-0.193937,7.782887,NaN,NaN
2,bangalore,Q1 2016,114.6000,71.406332,0.092739,0.144372,-0.051633,7.657853,0.237475,0.043538
3,bangalore,Q1 2017,121.9875,75.582470,0.062471,0.056838,0.005633,6.962920,0.144372,0.092739
4,bangalore,Q1 2018,145.2250,77.997789,0.174366,0.031456,0.142910,7.466353,0.056838,0.062471
5,bangalore,Q1 2019,144.7375,83.885498,-0.003363,0.072772,-0.076135,7.397403,0.031456,0.174366


In [32]:
display(final_model_dataset.describe())

,rent,hpi,rent_growth,hpi_growth,valuation_gap,repo_rate,hpi_lag,rent_lag
count,184.000000,184.000000,184.000000,184.000000,184.000000,176.000000,180.000000,180.000000
mean,129.392294,91.558634,0.009141,0.015010,-0.005869,7.138252,0.013307,0.007684
std,24.436379,18.418792,0.130952,0.193391,0.126040,0.643533,0.194455,0.131900
min,94.500000,51.383168,-0.666421,-0.980377,-0.478007,5.910980,-0.980377,-0.666421
25%,109.937500,77.082516,-0.011665,-0.005891,-0.073576,6.714939,-0.006075,-0.015510
50%,123.029464,93.775282,0.034746,0.052051,-0.021541,7.136827,0.052051,0.034148
75%,143.553125,102.947493,0.072817,0.104940,0.056303,7.514228,0.104940,0.070054
max,203.675000,146.223219,0.277253,0.275671,0.511339,8.791340,0.259346,0.277253


In [33]:
count_hpi_growth_filtered = final_model_dataset[final_model_dataset["hpi_growth"].abs() < 0.5].shape[0]
print(f"Total count of records where absolute hpi_growth is less than 0.5: {count_hpi_growth_filtered}")

Total count of records where absolute hpi_growth is less than 0.5: 175


In [34]:
records_high_hpi_growth = final_model_dataset[final_model_dataset["hpi_growth"].abs() >= 0.5]
print(f"Number of records with absolute HPI growth >= 0.5: {records_high_hpi_growth.shape[0]}")
display(records_high_hpi_growth)

Number of records with absolute HPI growth >= 0.5: 9


,city,quarter,rent,hpi,rent_growth,hpi_growth,valuation_gap,repo_rate,hpi_lag,rent_lag
12,bangalore,Q2 2014,108.275000,51.519420,-0.567730,-0.799748,0.232018,8.791340,0.114417,0.004657
24,bangalore,Q3 2014,99.537500,57.989084,-0.666421,-0.668562,0.002141,8.569230,0.104768,0.015992
36,bangalore,Q4 2014,109.887500,63.138094,-0.617068,-0.597512,-0.019557,8.181552,0.012016,0.063140
59,chennai,Q2 2014,103.600000,51.383168,-0.469038,-0.980377,0.511339,8.791340,0.204256,0.017667
71,chennai,Q3 2014,100.100000,59.708753,-0.480809,-0.886051,0.405242,8.569230,0.225135,-0.037580
83,chennai,Q4 2014,111.000000,61.418491,-0.356855,-0.867423,0.510568,8.181552,0.160679,-0.061737
153,mumbai,Q2 2014,102.371429,57.965219,-0.302366,-0.601029,0.298663,8.791340,0.086324,0.080842
165,mumbai,Q3 2014,107.242857,57.045291,-0.289545,-0.604990,0.315445,8.569230,0.020201,0.096338
177,mumbai,Q4 2014,105.314286,60.110635,-0.350443,-0.562960,0.212517,8.181552,0.040572,0.128385


In [35]:
final_model_dataset = final_model_dataset[final_model_dataset["hpi_growth"].abs() < 0.5]
print(f"Shape of final_model_dataset after removing high HPI growth records: {final_model_dataset.shape}")
display(final_model_dataset.head())

Shape of final_model_dataset after removing high HPI growth records: (175, 10)


,city,quarter,rent,hpi,rent_growth,hpi_growth,valuation_gap,repo_rate,hpi_lag,rent_lag
1,bangalore,Q1 2015,104.4500,61.806841,0.043538,0.237475,-0.193937,7.782887,NaN,NaN
2,bangalore,Q1 2016,114.6000,71.406332,0.092739,0.144372,-0.051633,7.657853,0.237475,0.043538
3,bangalore,Q1 2017,121.9875,75.582470,0.062471,0.056838,0.005633,6.962920,0.144372,0.092739
4,bangalore,Q1 2018,145.2250,77.997789,0.174366,0.031456,0.142910,7.466353,0.056838,0.062471
5,bangalore,Q1 2019,144.7375,83.885498,-0.003363,0.072772,-0.076135,7.397403,0.031456,0.174366


In [36]:
print("Summary Statistics for final_model_dataset:")
display(final_model_dataset.describe())

Summary Statistics for final_model_dataset:


,rent,hpi,rent_growth,hpi_growth,valuation_gap,repo_rate,hpi_lag,rent_lag
count,175.000000,175.000000,175.000000,175.000000,175.000000,167.000000,171.000000,171.000000
mean,130.633449,93.294346,0.033041,0.053317,-0.020276,7.064108,0.008345,0.006289
std,24.404011,17.149197,0.073448,0.089757,0.103587,0.570082,0.197581,0.134513
min,94.500000,59.203005,-0.273759,-0.300430,-0.478007,5.910980,-0.980377,-0.666421
25%,110.600000,78.777559,-0.002539,0.000469,-0.078284,6.676732,-0.007579,-0.016117
50%,124.300000,95.690968,0.038898,0.061195,-0.024529,7.119050,0.050597,0.034501
75%,144.256250,103.979883,0.074434,0.105878,0.046676,7.405995,0.103158,0.069209
max,203.675000,146.223219,0.277253,0.275671,0.230875,8.791340,0.259346,0.277253


In [37]:
records_high_rent_lag = final_model_dataset[final_model_dataset["rent_lag"].abs() > 0.5]
print(f"Number of records with absolute rent_lag > 0.5: {records_high_rent_lag.shape[0]}")
display(records_high_rent_lag)

Number of records with absolute rent_lag > 0.5: 3


,city,quarter,rent,hpi,rent_growth,hpi_growth,valuation_gap,repo_rate,hpi_lag,rent_lag
13,bangalore,Q2 2015,108.6250,59.819703,0.003227,0.149376,-0.146149,7.831457,-0.799748,-0.567730
25,bangalore,Q3 2015,113.0125,62.725676,0.126964,0.078516,0.048448,7.778315,-0.668562,-0.666421
37,bangalore,Q4 2015,122.9875,69.447592,0.112626,0.095248,0.017378,7.711225,-0.597512,-0.617068


In [38]:
records_high_repo_rate = final_model_dataset[final_model_dataset["repo_rate"].abs() > 0.5]
print(f"Number of records with absolute repo_rate > 0.5: {records_high_repo_rate.shape[0]}")
display(records_high_repo_rate)

Number of records with absolute repo_rate > 0.5: 167


,city,quarter,rent,hpi,rent_growth,hpi_growth,valuation_gap,repo_rate,hpi_lag,rent_lag
1,bangalore,Q1 2015,104.450000,61.806841,0.043538,0.237475,-0.193937,7.782887,NaN,NaN
2,bangalore,Q1 2016,114.600000,71.406332,0.092739,0.144372,-0.051633,7.657853,0.237475,0.043538
3,bangalore,Q1 2017,121.987500,75.582470,0.062471,0.056838,0.005633,6.962920,0.144372,0.092739
4,bangalore,Q1 2018,145.225000,77.997789,0.174366,0.031456,0.142910,7.466353,0.056838,0.062471
5,bangalore,Q1 2019,144.737500,83.885498,-0.003363,0.072772,-0.076135,7.397403,0.031456,0.174366
...,...,...,...,...,...,...,...,...,...,...
183,mumbai,Q4 2020,110.657143,90.592899,-0.010147,0.011342,-0.021489,5.910980,0.016050,0.063304
184,mumbai,Q4 2021,108.457143,91.307325,-0.020082,0.007855,-0.027937,6.384825,0.011342,-0.010147
185,mumbai,Q4 2022,114.485714,96.374888,0.054095,0.054015,0.000080,7.372220,0.007855,-0.020082
186,mumbai,Q4 2023,124.600000,100.695009,0.084659,0.043851,0.040808,7.281333,0.054015,0.054095


In [39]:
final_model_dataset[final_model_dataset["quarter"].isin(["Q4 2014","Q1 2015"])][
    ["city","rent","hpi","rent_growth","hpi_growth"]
]

,city,rent,hpi,rent_growth,hpi_growth
1,bangalore,104.450000,61.806841,0.043538,0.237475
48,chennai,111.100000,61.395709,0.105261,0.259346
95,delhi,131.950000,82.808833,0.277253,0.118023
130,delhi,105.150000,78.557836,-0.256163,-0.230440
142,mumbai,99.828571,62.769465,-0.001716,0.135612


In [40]:
final_model_dataset[['rent_growth','hpi_growth']].corr()

,rent_growth,hpi_growth
rent_growth,1.000000,0.206341
hpi_growth,0.206341,1.000000


### 9. Output Final Dataset

- Save the final dataset as `final_model_dataset.csv`.
- Print the first 5 rows of the saved dataset.

In [41]:
# Save final dataset
output_filename = 'final_model_dataset.csv'
final_model_dataset.to_csv(f'/content/{output_filename}', index=False)

print(f"\nFinal dataset saved as '/content/{output_filename}'.")

print("\nFirst 5 rows of the final dataset:")
display(final_model_dataset.head())
print(f"Shape of final_model_dataset: {final_model_dataset.shape}")


Final dataset saved as '/content/final_model_dataset.csv'.

First 5 rows of the final dataset:


,city,quarter,rent,hpi,rent_growth,hpi_growth,valuation_gap,repo_rate,hpi_lag,rent_lag
1,bangalore,Q1 2015,104.4500,61.806841,0.043538,0.237475,-0.193937,7.782887,NaN,NaN
2,bangalore,Q1 2016,114.6000,71.406332,0.092739,0.144372,-0.051633,7.657853,0.237475,0.043538
3,bangalore,Q1 2017,121.9875,75.582470,0.062471,0.056838,0.005633,6.962920,0.144372,0.092739
4,bangalore,Q1 2018,145.2250,77.997789,0.174366,0.031456,0.142910,7.466353,0.056838,0.062471
5,bangalore,Q1 2019,144.7375,83.885498,-0.003363,0.072772,-0.076135,7.397403,0.031456,0.174366


Shape of final_model_dataset: (175, 10)
